# Huấn luyện Mô hình CRAM-ABSA


In [ ]:
import os
import re
import sys
import math
import json
import time
import shutil
import pickle
import random
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from typing import List, Dict, Tuple, Optional, Any

os.environ["TOKENIZERS_PARALLELISM"] = "false"

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel, AutoConfig, get_cosine_schedule_with_warmup
from sklearn.metrics import f1_score, mean_absolute_error, accuracy_score, cohen_kappa_score

import nltk
for pkg in ["stopwords", "punkt", "averaged_perceptron_tagger", "wordnet"]:
    try:
        nltk.data.find(f"corpora/{pkg}" if pkg in ["stopwords", "wordnet"] else f"tokenizers/{pkg}")
    except LookupError:
        try:
            nltk.download(pkg, quiet=True)
        except Exception:
            pass

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

print("=" * 80)
print(f"PyTorch Version: {torch.__version__} | Device: {DEVICE} | GPU Count: {NUM_GPUS}")
total_vram_gb = 0.0
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    vram_gb = props.total_memory / (1024 ** 3)
    total_vram_gb += vram_gb
    print(f"  [GPU {i}] {props.name} | VRAM: {vram_gb:.2f} GB | Compute Capability: {props.major}.{props.minor}")
print(f"Tổng dung lượng VRAM khả dụng cho huấn luyện CRAM-ABSA: {total_vram_gb:.2f} GB")
print("=" * 80)


## 1. Cấu hình Siêu tham số


In [ ]:
class TrainConfig:
    PARQUET_CANDIDATES = [
        "/kaggle/input/dap_dts/hotel_reviews.parquet",
        "/kaggle/input/dap-dts/hotel_reviews.parquet",
        "/kaggle/input/hotel-reviews-cram-absa/hotel_reviews.parquet",
        "/kaggle/input/tripadvisor-hotel-reviews/hotel_reviews.parquet",
        "data/df/hotel_reviews.parquet",
        "hotel_reviews.parquet"
    ]
    OUTPUT_DIR = Path("/kaggle/working" if Path("/kaggle/working").exists() else "model")
    LOCAL_MODEL_DIR = Path("model")

    MODEL_FILENAME = "modelv1.pt"
    MODEL_SAVE_PATH = OUTPUT_DIR / "modelv1.pt"
    LOCAL_MODEL_SAVE_PATH = LOCAL_MODEL_DIR / "modelv1.pt"
    EBR_BANK_SAVE_PATH = OUTPUT_DIR / "ebr_exemplar_bank.pkl"
    HISTORY_SAVE_PATH = OUTPUT_DIR / "training_history_v1.json"

    BACKBONE_NAME = "bert-base-uncased"
    HIDDEN_DIM = 768
    NUM_FEATURES = 20
    PROJ_DIM = 256
    CANONICAL_ASPECTS = ["Service", "Facility", "Experience", "Amenity", "Loyalty", "Branding"]
    NUM_ASPECTS = 6
    NUM_SENTIMENT_CLASSES = 4
    NUM_STARS = 5
    MAX_SEQ_LEN = 256
    MAX_COMMENT_LEN = 48

    PER_GPU_BATCH_SIZE = 32
    BATCH_SIZE = PER_GPU_BATCH_SIZE * max(1, NUM_GPUS)
    GRAD_ACCUM_STEPS = 2
    NUM_WORKERS = 2 if os.name != "nt" else 0
    PIN_MEMORY = torch.cuda.is_available()
    USE_AMP_FP16 = torch.cuda.is_available()

    EPOCHS = 8
    LR_BACKBONE = 3e-5
    LR_HEADS = 1e-4
    WEIGHT_DECAY = 0.01
    WARMUP_RATIO = 0.10
    LABEL_SMOOTHING = 0.10
    MAX_GRAD_NORM = 1.0

    BETA_CORN = 0.30
    BETA_OVERLAP = 0.15
    BETA_RATIONALE = 0.25
    BETA_COMMENTARY = 0.35

    FEATURE_COLS_20 = [
        "feat_aspect_count",
        "feat_dominant_aspect",
        "feat_net_sentiment_score",
        "feat_sentiment_conflict",
        "feat_evidence_density",
        "feat_avg_rationale_len",
        "feat_has_cross_overlap",
        "feat_service_sentiment",
        "feat_facility_sentiment",
        "feat_experience_sentiment",
        "feat_amenity_sentiment",
        "feat_loyalty_sentiment",
        "feat_branding_sentiment",
        "feat_aspect_coverage_ratio",
        "feat_recency_weight",
        "feat_consensus_strength",
        "feat_temporal_aspect_sentiment",
        "feat_nltk_clean_token_ratio",
        "feat_nltk_opinion_negation_density",
        "feat_ebr_aspect_prototype_sim"
    ]

cfg = TrainConfig()
cfg.OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
cfg.LOCAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f"Model save path: {cfg.MODEL_SAVE_PATH}")


## 2. Tiền xử lý Ngôn ngữ Tự nhiên và Trích xuất 20 Features


In [ ]:
try:
    from nltk.corpus import stopwords
    RAW_NLTK_STOPWORDS = set(stopwords.words("english"))
except Exception:
    RAW_NLTK_STOPWORDS = {
        "a", "an", "the", "and", "or", "if", "in", "on", "at", "to", "from", "by", "for",
        "with", "about", "as", "into", "like", "through", "after", "over", "between", "out",
        "of", "during", "before", "under", "around", "among", "is", "are", "was", "were",
        "be", "been", "being", "have", "has", "had", "do", "does", "did", "will", "would",
        "shall", "should", "can", "could", "may", "might", "must", "i", "me", "my", "myself",
        "we", "our", "ours", "ourselves", "you", "your", "yours", "he", "him", "his", "she",
        "her", "hers", "it", "its", "they", "them", "their", "theirs", "this", "that", "these",
        "those", "am", "so", "than", "too", "very", "just", "where", "when", "who", "whom",
        "which", "what", "there", "here", "all", "any", "both", "each", "few", "more", "most",
        "other", "some", "such", "only", "own", "same", "then", "once"
    }

CONTRACTIONS_MAP = {
    "can't": "cannot", "won't": "will not", "don't": "do not", "doesn't": "does not",
    "didn't": "did not", "isn't": "is not", "aren't": "are not", "wasn't": "was not",
    "weren't": "were not", "haven't": "have not", "hasn't": "has not", "hadn't": "had not",
    "couldn't": "could not", "wouldn't": "would not", "shouldn't": "should not",
    "n't": " not", "'re": " are", "'ve": " have", "'ll": " will", "'m": " am"
}

NEGATION_AND_CONTRAST_WORDS = {
    "not", "no", "never", "neither", "nor", "none", "nothing", "nowhere", "hardly",
    "barely", "scarcely", "without", "but", "however", "although", "though", "yet",
    "despite", "unfortunately", "except", "fault"
}

SAFE_STOPWORDS = RAW_NLTK_STOPWORDS - NEGATION_AND_CONTRAST_WORDS

OPINION_LEXICON_POS = {
    "good", "great", "excellent", "amazing", "wonderful", "friendly", "helpful", "clean",
    "spacious", "comfortable", "delicious", "lovely", "perfect", "best", "nice", "fantastic",
    "superb", "polite", "attentive", "convenient", "beautiful", "cozy", "relaxing", "awesome",
    "recommend", "recommended", "happy", "pleased", "impressed", "welcoming", "professional",
    "quiet", "modern", "fresh", "tasty", "worth", "luxury", "luxurious", "outstanding", "warm"
}

OPINION_LEXICON_NEG = {
    "bad", "poor", "terrible", "horrible", "awful", "slow", "rude", "dirty", "noisy", "small",
    "old", "outdated", "broken", "cold", "hard", "uncomfortable", "overpriced", "expensive",
    "worst", "disappointed", "disappointing", "unfriendly", "unhelpful", "smell", "smelly",
    "stain", "stained", "cockroach", "bug", "mold", "issue", "problem", "complaint", "wait",
    "waiting", "delayed", "mediocre", "basic", "boring", "crowded", "weak", "fault"
}

INTENSIFIERS = {
    "very", "extremely", "really", "super", "highly", "totally", "absolutely", "completely",
    "quite", "pretty", "so", "too", "exceptionally", "incredibly", "amazingly", "remarkably"
}

ASPECT_KEYWORDS = {
    "Service": {"staff", "service", "reception", "front", "desk", "manager", "waiter", "waitress", "concierge", "bellboy", "helpful", "friendly", "polite", "check", "checkin", "checkout", "attendant", "housekeeping", "host", "crew"},
    "Facility": {"room", "bed", "bathroom", "shower", "toilet", "air", "ac", "conditioning", "clean", "cleanliness", "noise", "noisy", "soundproof", "view", "balcony", "floor", "elevator", "lift", "pillow", "mattress", "furniture", "decor", "water", "carpet"},
    "Experience": {"stay", "stayed", "experience", "holiday", "vacation", "trip", "night", "overall", "atmosphere", "vibe", "relax", "relaxing", "enjoy", "enjoyed", "memorable", "time", "comfort", "comfortable", "quiet", "peaceful", "feeling"},
    "Amenity": {"breakfast", "buffet", "pool", "swimming", "spa", "gym", "fitness", "wifi", "internet", "restaurant", "bar", "food", "drink", "coffee", "tea", "lounge", "shuttle", "parking", "massage", "rooftop"},
    "Loyalty": {"back", "return", "again", "recommend", "recommended", "definitely", "choose", "future", "next", "visit", "repeat", "choice", "staying"},
    "Branding": {"star", "stars", "luxury", "brand", "hotel", "resort", "standard", "reputation", "class", "five", "four", "chain", "management", "value", "price", "worth", "quality", "prestigious", "accor", "sofitel", "pullman", "novotel", "marriott", "hilton", "intercontinental", "sheraton"}
}

ASPECT_ID_MAP = {asp: idx + 1 for idx, asp in enumerate(cfg.CANONICAL_ASPECTS)}
ASPECT_ID_MAP["None"] = 0

def nltk_negation_aware_preprocess(text: str) -> Tuple[List[str], List[str]]:
    lower_txt = unicodedata.normalize("NFC", str(text)).lower()
    for k, v in CONTRACTIONS_MAP.items():
        lower_txt = re.sub(r"\b" + re.escape(k) + r"\b", v, lower_txt)
        lower_txt = lower_txt.replace(k, v)
    lower_txt = re.sub(r"@\w+|https?://\S+|www\.\S+", " ", lower_txt)
    raw_tokens = re.findall(r"[a-z]+", lower_txt)
    clean_tokens = [t for t in raw_tokens if t not in SAFE_STOPWORDS and len(t) > 1]
    return raw_tokens, clean_tokens

def preprocess_text_paper1(text: str) -> Tuple[str, str]:

    norm_txt = unicodedata.normalize("NFC", str(text)).lower()
    for k, v in CONTRACTIONS_MAP.items():
        norm_txt = re.sub(r"\b" + re.escape(k) + r"\b", v, norm_txt)
        norm_txt = norm_txt.replace(k, v)
    norm_txt = re.sub(r"@\w+|https?://\S+|www\.\S+", " ", norm_txt)
    clean_review_text = re.sub(r"[^a-z0-9.,!?'\s-]", " ", norm_txt)
    clean_review_text = re.sub(r"\s+", " ", clean_review_text).strip()
    _, clean_tokens = nltk_negation_aware_preprocess(norm_txt)
    paper1_nltk_text = " ".join(clean_tokens)
    return clean_review_text, paper1_nltk_text

def parse_annotations_safe(anns: Any) -> List[Dict[str, Any]]:

    if anns is None:
        return []
    if isinstance(anns, np.ndarray):
        anns = anns.tolist()
    elif isinstance(anns, str):
        try:
            anns = json.loads(anns)
        except Exception:
            return []
    if not isinstance(anns, list):
        return []
    return [dict(a) for a in anns if isinstance(a, dict)]

def extract_20_features_for_new_review(review_text: str, recency_weight: float = 1.0) -> np.ndarray:

    raw_text = unicodedata.normalize("NFC", str(review_text).strip())
    char_len = max(len(raw_text), 1)
    raw_tokens, clean_tokens = nltk_negation_aware_preprocess(raw_text)
    total_raw_toks = max(len(raw_tokens), 1)

    clean_token_ratio = float(min(len(clean_tokens) / total_raw_toks, 1.0))
    opinion_neg_cnt = sum(
        1 for t in raw_tokens
        if (t in OPINION_LEXICON_POS or t in OPINION_LEXICON_NEG or t in NEGATION_AND_CONTRAST_WORDS or t in INTENSIFIERS)
    )
    opinion_neg_density = float(min(opinion_neg_cnt / total_raw_toks, 1.0))

    tok_counts = Counter(clean_tokens)
    aspect_sims = []
    for asp in cfg.CANONICAL_ASPECTS:
        kw_set = ASPECT_KEYWORDS[asp]
        overlap_w = sum(cnt for tok, cnt in tok_counts.items() if tok in kw_set)
        norm_sim = overlap_w / math.sqrt(len(clean_tokens) * len(kw_set)) if clean_tokens else 0.0
        aspect_sims.append(min(1.0, norm_sim * 3.5))
    top_sims = sorted(aspect_sims, reverse=True)[:3]
    ebr_proto_sim = float(sum(top_sims) / len(top_sims)) if top_sims else 0.0

    sentences = [s.strip() for s in re.split(r"[.!?]+", raw_text) if s.strip()]
    aspect_scores = {}
    matched_sentences_per_aspect = defaultdict(list)
    overlap_flag = 0

    for sent in sentences:
        s_raw, s_clean = nltk_negation_aware_preprocess(sent)
        matched_asps = [asp for asp in cfg.CANONICAL_ASPECTS if any(t in ASPECT_KEYWORDS[asp] for t in s_clean)]
        if len(matched_asps) >= 2:
            overlap_flag = 1
        pos_hits = sum(1 for t in s_raw if t in OPINION_LEXICON_POS)
        neg_hits = sum(1 for t in s_raw if t in OPINION_LEXICON_NEG)
        for idx_t, tok in enumerate(s_raw[:-1]):
            if tok in {"not", "never", "no", "hardly"}:
                nxt = s_raw[idx_t + 1]
                if nxt in OPINION_LEXICON_POS:
                    pos_hits = max(0, pos_hits - 1)
                    neg_hits += 1
                elif nxt in OPINION_LEXICON_NEG:
                    neg_hits = max(0, neg_hits - 1)
                    pos_hits += 1
        sent_pol = (pos_hits - neg_hits) / (pos_hits + neg_hits) if (pos_hits + neg_hits) > 0 else 0.25
        for asp in matched_asps:
            matched_sentences_per_aspect[asp].append((sent, sent_pol))

    for asp in cfg.CANONICAL_ASPECTS:
        items = matched_sentences_per_aspect.get(asp, [])
        if items:
            aspect_scores[asp] = float(np.clip(np.mean([sc for _, sc in items]), -1.0, 1.0))
        else:
            aspect_scores[asp] = 0.0

    active_aspects = [asp for asp in cfg.CANONICAL_ASPECTS if matched_sentences_per_aspect.get(asp)]
    asp_count = len(active_aspects)
    dominant_aspect = ASPECT_ID_MAP.get(active_aspects[0], 0) if active_aspects else 0
    active_vals = [aspect_scores[a] for a in active_aspects] if active_aspects else [0.0]
    net_sentiment = float(np.mean(active_vals))
    sent_conflict = 1 if (any(v > 0.1 for v in active_vals) and any(v < -0.1 for v in active_vals)) else 0
    all_matched_Lens = [len(s) for items in matched_sentences_per_aspect.values() for s, _ in items]
    evidence_density = float(min(sum(all_matched_Lens) / char_len, 1.0)) if all_matched_Lens else clean_token_ratio
    avg_rat_len = float(np.mean(all_matched_Lens)) if all_matched_Lens else float(char_len / max(asp_count, 1))
    coverage_ratio = float(min(asp_count / 6.0, 1.0))
    consensus_strength = float(abs(net_sentiment))
    temporal_aspect_sentiment = float(recency_weight * net_sentiment)

    vec_20 = np.array([
        asp_count,
        dominant_aspect,
        net_sentiment,
        sent_conflict,
        evidence_density,
        avg_rat_len,
        overlap_flag,
        aspect_scores["Service"],
        aspect_scores["Facility"],
        aspect_scores["Experience"],
        aspect_scores["Amenity"],
        aspect_scores["Loyalty"],
        aspect_scores["Branding"],
        coverage_ratio,
        recency_weight,
        consensus_strength,
        temporal_aspect_sentiment,
        clean_token_ratio,
        opinion_neg_density,
        ebr_proto_sim
    ], dtype=np.float32)
    return vec_20

print("Đã khởi tạo thành công bộ xử lý Dual-Stream NLTK và hàm trích xuất 20 features!")


## 3. Nạp Dữ liệu Parquet và Chuẩn hóa Features


In [ ]:
parquet_path = None
for p in cfg.PARQUET_CANDIDATES:
    if Path(p).exists():
        parquet_path = Path(p)
        break

if parquet_path is None and Path("/kaggle/input").exists():
    found_parquets = list(Path("/kaggle/input").rglob("hotel_reviews.parquet"))
    if found_parquets:
        parquet_path = found_parquets[0]

if parquet_path is None:
    raise FileNotFoundError(f"Không tìm thấy hotel_reviews.parquet tại {cfg.PARQUET_CANDIDATES} hoặc trong /kaggle/input/. Hãy kiểm tra lại dataset đã được nạp!")

t0 = time.time()
df_master = pd.read_parquet(parquet_path)
print(f"Đã nạp '{parquet_path}' trong {time.time() - t0:.3f}s | Kích thước: {df_master.shape[0]:,} dòng x {df_master.shape[1]} cột")

if "clean_review_text" not in df_master.columns or "paper1_nltk_text" not in df_master.columns:
    print("Đang áp dụng tiền xử lý văn bản theo phương pháp paper 1.pdf (clean_review_text & paper1_nltk_text)...")
    cleaned_pairs = [preprocess_text_paper1(txt) for txt in df_master["review_text"]]
    df_master["clean_review_text"] = [p[0] for p in cleaned_pairs]
    df_master["paper1_nltk_text"] = [p[1] for p in cleaned_pairs]

missing_feats = [c for c in cfg.FEATURE_COLS_20 if c not in df_master.columns]
if missing_feats:
    print(f"Phát hiện thiếu {len(missing_feats)} features ({missing_feats}), đang tự động tính toán bổ sung...")
    computed = np.vstack([
        extract_20_features_for_new_review(txt, float(rw))
        for txt, rw in zip(df_master["review_text"], df_master.get("feat_recency_weight", np.ones(len(df_master))))
    ])
    for idx_f, col_name in enumerate(cfg.FEATURE_COLS_20):
        if col_name not in df_master.columns:
            df_master[col_name] = computed[:, idx_f]
else:
    if df_master["feat_dominant_aspect"].dtype == object:
        df_master["feat_dominant_aspect"] = df_master["feat_dominant_aspect"].map(lambda x: ASPECT_ID_MAP.get(str(x), 0)).astype(np.float32)

if "ordinal_rating" not in df_master.columns:
    df_master["ordinal_rating"] = df_master["rating_5"].clip(1, 5).astype(int) - 1

df_train = df_master[df_master["split"] == "train"].reset_index(drop=True)
df_val = df_master[df_master["split"] == "val"].reset_index(drop=True)
df_test = df_master[df_master["split"] == "test"].reset_index(drop=True)

feat_mean = df_train[cfg.FEATURE_COLS_20].values.astype(np.float32).mean(axis=0)
feat_std = df_train[cfg.FEATURE_COLS_20].values.astype(np.float32).std(axis=0)
feat_std = np.where(feat_std < 1e-6, 1.0, feat_std)

print(f"Tập Train: {len(df_train):,} mẫu | Tập Val: {len(df_val):,} mẫu | Tập Test: {len(df_test):,} mẫu")
display(df_train[["review_id", "clean_review_text", "paper1_nltk_text"] + cfg.FEATURE_COLS_20[-3:]].head(3))


## 4. Kho Truy xuất Mẫu chuẩn EBR


In [ ]:
class EBRExemplarBank:

    def __init__(self, canonical_aspects: List[str]):
        self.aspects = canonical_aspects
        self.bank = {asp: {"Positive": [], "Neutral": [], "Negative": []} for asp in canonical_aspects}
        self.aspect_keyword_idf = {}

    def build_from_dataframe(self, df_tr: pd.DataFrame):
        doc_freq = Counter()
        total_spans = 0
        for _, row in df_tr.iterrows():
            rw = float(row.get("feat_recency_weight", 1.0))
            anns = parse_annotations_safe(row.get("aspect_annotations", []))
            for ann in anns:
                if not isinstance(ann, dict):
                    continue
                asp = ann.get("aspect")
                sent = ann.get("sentiment", "Neutral")
                span_txt = str(ann.get("span_text", "")).strip()
                if asp in self.bank and sent in self.bank[asp] and len(span_txt) >= 12:
                    _, clean_toks = nltk_negation_aware_preprocess(span_txt)
                    if clean_toks:
                        total_spans += 1
                        for t in set(clean_toks):
                            doc_freq[t] += 1
                        self.bank[asp][sent].append({
                            "span_text": span_txt,
                            "tokens": set(clean_toks),
                            "recency_weight": rw,
                            "hotel_name": str(row.get("clean_hotel_name", "Hotel"))
                        })
        for tok, df_c in doc_freq.items():
            self.aspect_keyword_idf[tok] = math.log((total_spans + 1.0) / (df_c + 1.0)) + 1.0

        for asp in self.aspects:
            for sent in ["Positive", "Neutral", "Negative"]:
                items = self.bank[asp][sent]
                items.sort(key=lambda x: x["recency_weight"] * min(len(x["tokens"]), 12), reverse=True)
                self.bank[asp][sent] = items[:300]

    def retrieve_exemplar(self, review_text: str, aspect: str, sentiment_label: str) -> Tuple[str, float]:
        _, query_toks = nltk_negation_aware_preprocess(review_text)
        query_set = set(query_toks) | ASPECT_KEYWORDS.get(aspect, set())
        candidates = self.bank.get(aspect, {}).get(sentiment_label, [])
        if not candidates:
            return "", 0.0
        best_span = candidates[0]["span_text"]
        best_score = 0.0
        for cand in candidates:
            inter = query_set.intersection(cand["tokens"])
            idf_overlap = sum(self.aspect_keyword_idf.get(t, 1.0) for t in inter)
            norm = math.sqrt(len(query_set) * max(len(cand["tokens"]), 1))
            sim = (idf_overlap / max(norm, 1.0)) * (0.7 + 0.3 * cand["recency_weight"])
            if sim > best_score:
                best_score = sim
                best_span = cand["span_text"]
        return best_span, float(min(best_score, 1.0))

ebr_bank = EBRExemplarBank(cfg.CANONICAL_ASPECTS)
ebr_bank.build_from_dataframe(df_train)
with open(cfg.EBR_BANK_SAVE_PATH, "wb") as f:
    pickle.dump(ebr_bank, f)

for asp in cfg.CANONICAL_ASPECTS:
    cnts = {s: len(ebr_bank.bank[asp][s]) for s in ["Positive", "Neutral", "Negative"]}
    print(f"  [EBR Bank] Aspect '{asp:<10}' -> Exemplars lưu trữ: {cnts}")
print(f"Đã lưu kho truy xuất EBR tại: {cfg.EBR_BANK_SAVE_PATH}")


## 5. Thiết kế Dataset và DataLoader


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(cfg.BACKBONE_NAME)
SENTIMENT_TO_ID = {"Negative": 1, "Neutral": 2, "Positive": 3}
ID_TO_SENTIMENT = {0: "Not Mentioned", 1: "Negative", 2: "Neutral", 3: "Positive"}

class CRAMABSADataset(Dataset):
    def __init__(self, df: pd.DataFrame, tokenizer, feat_mean: np.ndarray, feat_std: np.ndarray, max_len: int = 256, max_comment_len: int = 48):
        self.df = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.feat_mean = feat_mean
        self.feat_std = feat_std
        self.max_len = max_len
        self.max_comment_len = max_comment_len
        self.features_raw = self.df[cfg.FEATURE_COLS_20].values.astype(np.float32)
        self.features_norm = (self.features_raw - self.feat_mean) / self.feat_std

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        raw_text = unicodedata.normalize("NFC", str(row["review_text"]))
        feat_vec = self.features_norm[idx]

        enc = self.tokenizer(
            raw_text,
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_offsets_mapping=True,
            return_tensors="pt"
        )
        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)
        offsets = enc["offset_mapping"].squeeze(0).numpy()

        ordinal_rating = int(row.get("ordinal_rating", int(row["rating_5"]) - 1))
        ordinal_rating = max(0, min(4, ordinal_rating))

        aspect_sent_labels = torch.zeros(cfg.NUM_ASPECTS, dtype=torch.long)
        aspect_star_labels = torch.full((cfg.NUM_ASPECTS,), ordinal_rating, dtype=torch.long)
        rationale_masks = torch.zeros((cfg.NUM_ASPECTS, self.max_len), dtype=torch.float32)
        commentary_target_ids = torch.full(
            (cfg.NUM_ASPECTS, self.max_comment_len),
            self.tokenizer.pad_token_id,
            dtype=torch.long
        )

        anns = parse_annotations_safe(row.get("aspect_annotations", []))
        asp_sent_votes = defaultdict(list)
        asp_span_texts = defaultdict(list)

        for ann in anns:
            asp = ann.get("aspect")
            sent = ann.get("sentiment", "Neutral")
            if asp not in ASPECT_ID_MAP or asp == "None":
                continue
            asp_idx = ASPECT_ID_MAP[asp] - 1
            s_char = int(ann.get("start_char", 0))
            e_char = int(ann.get("end_char", 0))
            span_txt = str(ann.get("span_text", "")).strip()
            asp_sent_votes[asp_idx].append(SENTIMENT_TO_ID.get(sent, 2))
            if span_txt:
                asp_span_texts[asp_idx].append(span_txt)

            for t_i, (tok_s, tok_e) in enumerate(offsets):
                if tok_e > tok_s and tok_s < e_char and tok_e > s_char:
                    rationale_masks[asp_idx, t_i] = 1.0

        for asp_idx, votes in asp_sent_votes.items():
            maj_sent = Counter(votes).most_common(1)[0][0]
            aspect_sent_labels[asp_idx] = maj_sent
            if maj_sent == 3:
                asp_star = max(3, ordinal_rating)
            elif maj_sent == 2:
                asp_star = 2
            else:
                asp_star = min(1, ordinal_rating)
            aspect_star_labels[asp_idx] = asp_star

            joined_span = " ".join(asp_span_texts[asp_idx])
            if joined_span:
                span_enc = self.tokenizer(
                    joined_span,
                    max_length=self.max_comment_len,
                    padding="max_length",
                    truncation=True,
                    return_tensors="pt"
                )
                commentary_target_ids[asp_idx] = span_enc["input_ids"].squeeze(0)

        aspect_corn_targets = torch.zeros((cfg.NUM_ASPECTS, cfg.NUM_STARS - 1), dtype=torch.float32)
        for a_i in range(cfg.NUM_ASPECTS):
            for k in range(cfg.NUM_STARS - 1):
                if aspect_star_labels[a_i] > k:
                    aspect_corn_targets[a_i, k] = 1.0

        corn_targets = torch.tensor(
            [1.0 if ordinal_rating > k else 0.0 for k in range(cfg.NUM_STARS - 1)],
            dtype=torch.float32
        )

        return {
            "review_id": int(row["review_id"]),
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "dense_features": torch.tensor(feat_vec, dtype=torch.float32),
            "aspect_sent_labels": aspect_sent_labels,
            "aspect_star_labels": aspect_star_labels,
            "aspect_corn_targets": aspect_corn_targets,
            "rationale_masks": rationale_masks,
            "commentary_target_ids": commentary_target_ids,
            "ordinal_rating": torch.tensor(ordinal_rating, dtype=torch.long),
            "corn_targets": corn_targets
        }

train_dataset = CRAMABSADataset(df_train, tokenizer, feat_mean, feat_std, cfg.MAX_SEQ_LEN, cfg.MAX_COMMENT_LEN)
val_dataset = CRAMABSADataset(df_val, tokenizer, feat_mean, feat_std, cfg.MAX_SEQ_LEN, cfg.MAX_COMMENT_LEN)
test_dataset = CRAMABSADataset(df_test, tokenizer, feat_mean, feat_std, cfg.MAX_SEQ_LEN, cfg.MAX_COMMENT_LEN)

train_loader = DataLoader(
    train_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=True,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=cfg.PIN_MEMORY,
    drop_last=False
)
val_loader = DataLoader(
    val_dataset,
    batch_size=cfg.BATCH_SIZE,
    shuffle=False,
    num_workers=cfg.NUM_WORKERS,
    pin_memory=cfg.PIN_MEMORY
)

print(f"Train Batches: {len(train_loader)} | Val Batches: {len(val_loader)}")


## 6. Kiến trúc Mô hình CRAM-ABSA


In [ ]:
class CRAM_ABSA_MultiAspectModel(nn.Module):
    def __init__(self, cfg: TrainConfig):
        super().__init__()
        self.num_aspects = cfg.NUM_ASPECTS
        self.num_sent_classes = cfg.NUM_SENTIMENT_CLASSES
        self.hidden_dim = cfg.HIDDEN_DIM
        self.max_comment_len = getattr(cfg, "MAX_COMMENT_LEN", 48)

        self.bert = AutoModel.from_pretrained(cfg.BACKBONE_NAME)
        self.vocab_size = self.bert.config.vocab_size
        self.pad_token_id = self.bert.config.pad_token_id or 0

        self.feature_proj = nn.Sequential(
            nn.Linear(cfg.NUM_FEATURES, cfg.PROJ_DIM),
            nn.LayerNorm(cfg.PROJ_DIM),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(cfg.PROJ_DIM, cfg.HIDDEN_DIM),
            nn.LayerNorm(cfg.HIDDEN_DIM)
        )
        self.residual_gate = nn.Parameter(torch.tensor(0.25))

        self.ebr_aspect_prototypes = nn.Parameter(torch.randn(cfg.NUM_ASPECTS, cfg.HIDDEN_DIM) * 0.02)
        self.rationale_proj = nn.Linear(cfg.HIDDEN_DIM, cfg.HIDDEN_DIM)
        self.rationale_bias = nn.Parameter(torch.zeros(cfg.NUM_ASPECTS, 1))

        self.aspect_classifiers = nn.ModuleList([
            nn.Sequential(
                nn.Linear(cfg.HIDDEN_DIM * 2, cfg.PROJ_DIM),
                nn.LayerNorm(cfg.PROJ_DIM),
                nn.GELU(),
                nn.Dropout(0.20),
                nn.Linear(cfg.PROJ_DIM, cfg.NUM_SENTIMENT_CLASSES)
            )
            for _ in range(cfg.NUM_ASPECTS)
        ])

        self.aspect_star_heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(cfg.HIDDEN_DIM * 2, cfg.PROJ_DIM),
                nn.LayerNorm(cfg.PROJ_DIM),
                nn.GELU(),
                nn.Dropout(0.20),
                nn.Linear(cfg.PROJ_DIM, cfg.NUM_STARS - 1)
            )
            for _ in range(cfg.NUM_ASPECTS)
        ])

        self.corn_head = nn.Sequential(
            nn.Linear(cfg.HIDDEN_DIM * 2, cfg.PROJ_DIM),
            nn.LayerNorm(cfg.PROJ_DIM),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(cfg.PROJ_DIM, cfg.NUM_STARS - 1)
        )

        self.commentary_gru = nn.GRU(
            input_size=cfg.HIDDEN_DIM,
            hidden_size=cfg.HIDDEN_DIM,
            num_layers=1,
            batch_first=True
        )
        self.commentary_out = nn.Linear(cfg.HIDDEN_DIM, self.vocab_size, bias=False)
        self.commentary_out.weight = self.bert.embeddings.word_embeddings.weight

    def forward(self, input_ids, attention_mask, dense_features, commentary_target_ids=None):
        bert_out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        seq_hidden = bert_out.last_hidden_state
        cls_hidden = seq_hidden[:, 0, :]

        feat_emb = self.feature_proj(dense_features)
        conditioned_seq = seq_hidden + self.residual_gate * feat_emb.unsqueeze(1)
        conditioned_cls = cls_hidden + feat_emb

        token_proj = self.rationale_proj(conditioned_seq)
        proto_norm = F.normalize(self.ebr_aspect_prototypes, p=2, dim=-1)
        token_norm = F.normalize(token_proj, p=2, dim=-1)
        bilinear_scores = torch.einsum("bld,ad->bal", token_proj, self.ebr_aspect_prototypes) / math.sqrt(self.hidden_dim)
        cosine_scores = torch.einsum("bld,ad->bal", token_norm, proto_norm)
        rationale_logits = bilinear_scores + cosine_scores + self.rationale_bias

        pad_mask = attention_mask.unsqueeze(1).float()
        rationale_probs = torch.sigmoid(rationale_logits) * pad_mask

        aspect_logits_list = []
        aspect_star_logits_list = []
        aspect_context_list = []
        denom = rationale_probs.sum(dim=-1, keepdim=True).clamp(min=1e-4)
        masked_pooled = torch.einsum("bal,bld->bad", rationale_probs, conditioned_seq) / denom

        for a_idx in range(self.num_aspects):
            asp_ctx = torch.cat([masked_pooled[:, a_idx, :], feat_emb], dim=-1)
            aspect_logits_list.append(self.aspect_classifiers[a_idx](asp_ctx))
            aspect_star_logits_list.append(self.aspect_star_heads[a_idx](asp_ctx))
            aspect_context_list.append(masked_pooled[:, a_idx, :])

        aspect_logits = torch.stack(aspect_logits_list, dim=1)
        aspect_star_logits = torch.stack(aspect_star_logits_list, dim=1)

        global_aspect_ctx = torch.stack(aspect_context_list, dim=1).mean(dim=1)
        corn_input = torch.cat([conditioned_cls, global_aspect_ctx], dim=-1)
        corn_logits = self.corn_head(corn_input)

        commentary_logits = None
        if commentary_target_ids is not None:
            bsz, n_asp, c_len = commentary_target_ids.size()
            flat_targets = commentary_target_ids.view(bsz * n_asp, c_len)
            dec_in_ids = flat_targets[:, :-1]
            dec_emb = self.bert.embeddings.word_embeddings(dec_in_ids)
            init_h = masked_pooled.view(1, bsz * n_asp, self.hidden_dim).contiguous()
            dec_out, _ = self.commentary_gru(dec_emb, init_h)
            commentary_logits = self.commentary_out(dec_out).view(bsz, n_asp, c_len - 1, self.vocab_size)

        return {
            "aspect_logits": aspect_logits,
            "aspect_star_logits": aspect_star_logits,
            "rationale_logits": rationale_logits,
            "rationale_probs": rationale_probs,
            "corn_logits": corn_logits,
            "commentary_logits": commentary_logits,
            "aspect_embeddings": masked_pooled,
            "seq_hidden": conditioned_seq
        }

    @torch.no_grad()
    def generate_aspect_commentary(self, aspect_idx: int, outputs: Dict[str, torch.Tensor], input_ids: torch.Tensor, attention_mask: torch.Tensor, tokenizer) -> str:
        rat_prob = outputs["rationale_probs"][0, aspect_idx]
        valid_len = int(attention_mask[0].sum().item())
        tok_ids = input_ids[0, :valid_len]
        scores = rat_prob[:valid_len]

        h_state = outputs["aspect_embeddings"][:, aspect_idx, :].unsqueeze(0).contiguous()
        cur_tok = torch.tensor([[tokenizer.cls_token_id]], dtype=torch.long, device=input_ids.device)
        gen_ids = []

        allowed_tokens = set(tok_ids.tolist()) | {tokenizer.sep_token_id, tokenizer.pad_token_id}
        vocab_mask = torch.full((self.vocab_size,), -1e4, device=input_ids.device)
        for tid in allowed_tokens:
            vocab_mask[tid] = 0.0

        for _ in range(self.max_comment_len - 1):
            emb = self.bert.embeddings.word_embeddings(cur_tok)
            dec_out, h_state = self.commentary_gru(emb, h_state)
            step_logits = self.commentary_out(dec_out[:, -1, :]).squeeze(0) + vocab_mask
            for prev_id in gen_ids[-4:]:
                step_logits[prev_id] -= 3.0
            next_id = int(torch.argmax(step_logits).item())
            if next_id in (tokenizer.sep_token_id, tokenizer.pad_token_id, tokenizer.cls_token_id):
                break
            gen_ids.append(next_id)
            cur_tok = torch.tensor([[next_id]], dtype=torch.long, device=input_ids.device)

        best_window_ids = []
        if valid_len > 2:
            win_size = min(18, valid_len - 2)
            best_score = -1.0
            best_start = 1
            for s_i in range(1, max(2, valid_len - win_size)):
                e_i = min(valid_len - 1, s_i + win_size)
                w_sc = float(scores[s_i:e_i].mean().item())
                if w_sc > best_score:
                    best_score = w_sc
                    best_start = s_i
            best_window_ids = tok_ids[best_start:min(valid_len - 1, best_start + win_size)].tolist()

        decoded_gen = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
        decoded_span = tokenizer.decode(best_window_ids, skip_special_tokens=True).strip()
        return decoded_span if len(decoded_span) >= len(decoded_gen) else decoded_gen


## 7. Hàm Mất mát Đa nhiệm


In [ ]:
class CRAMABSALoss(nn.Module):
    def __init__(self, cfg: TrainConfig, class_weights_corn: torch.Tensor, pad_token_id: int = 0):
        super().__init__()
        self.cfg = cfg
        self.register_buffer("corn_weights", class_weights_corn)
        self.aspect_ce = nn.CrossEntropyLoss(label_smoothing=cfg.LABEL_SMOOTHING)
        self.commentary_ce = nn.CrossEntropyLoss(ignore_index=pad_token_id)

    def compute_corn_loss(self, corn_logits: torch.Tensor, corn_targets: torch.Tensor, ordinal_rating: torch.Tensor) -> torch.Tensor:
        bce = F.binary_cross_entropy_with_logits(corn_logits, corn_targets, reduction="none")
        sample_w = self.corn_weights[ordinal_rating].unsqueeze(-1)
        return (bce * sample_w).mean()

    def compute_overlap_penalty(self, rationale_probs: torch.Tensor) -> torch.Tensor:
        norm_masks = F.normalize(rationale_probs, p=2, dim=-1, eps=1e-6)
        gram_matrix = torch.bmm(norm_masks, norm_masks.transpose(1, 2))
        eye = torch.eye(rationale_probs.size(1), device=rationale_probs.device).unsqueeze(0)
        off_diag = gram_matrix * (1.0 - eye)
        num_pairs = rationale_probs.size(1) * (rationale_probs.size(1) - 1)
        return off_diag.sum(dim=(1, 2)).mean() / max(num_pairs, 1)

    def forward(self, outputs: Dict[str, torch.Tensor], batch: Dict[str, torch.Tensor]) -> Dict[str, torch.Tensor]:
        aspect_logits = outputs["aspect_logits"]
        aspect_star_logits = outputs["aspect_star_logits"]
        rationale_logits = outputs["rationale_logits"]
        rationale_probs = outputs["rationale_probs"]
        corn_logits = outputs["corn_logits"]
        commentary_logits = outputs.get("commentary_logits")

        aspect_targets = batch["aspect_sent_labels"]
        aspect_corn_targets = batch["aspect_corn_targets"]
        rationale_targets = batch["rationale_masks"]
        corn_targets = batch["corn_targets"]
        ordinal_rating = batch["ordinal_rating"]
        pad_mask = batch["attention_mask"].unsqueeze(1).float()

        loss_aspect = self.aspect_ce(aspect_logits.view(-1, self.cfg.NUM_SENTIMENT_CLASSES), aspect_targets.view(-1))
        loss_aspect_star = F.binary_cross_entropy_with_logits(aspect_star_logits, aspect_corn_targets)
        loss_corn = self.compute_corn_loss(corn_logits, corn_targets, ordinal_rating)
        loss_overlap = self.compute_overlap_penalty(rationale_probs)

        pos_w = torch.tensor(4.0, device=rationale_logits.device)
        bce_rat = F.binary_cross_entropy_with_logits(
            rationale_logits, rationale_targets, pos_weight=pos_w, reduction="none"
        )
        loss_rationale = (bce_rat * pad_mask).sum() / pad_mask.sum().clamp(min=1.0)

        loss_commentary = torch.tensor(0.0, device=aspect_logits.device)
        if commentary_logits is not None and "commentary_target_ids" in batch:
            target_seq = batch["commentary_target_ids"][:, :, 1:].contiguous()
            loss_commentary = self.commentary_ce(
                commentary_logits.view(-1, commentary_logits.size(-1)),
                target_seq.view(-1)
            )

        total_loss = (
            loss_aspect
            + self.cfg.BETA_CORN * (loss_corn + loss_aspect_star)
            + self.cfg.BETA_OVERLAP * loss_overlap
            + self.cfg.BETA_RATIONALE * loss_rationale
            + self.cfg.BETA_COMMENTARY * loss_commentary
        )
        return {
            "total_loss": total_loss,
            "loss_aspect": loss_aspect.detach(),
            "loss_corn": loss_corn.detach(),
            "loss_overlap": loss_overlap.detach(),
            "loss_rationale": loss_rationale.detach(),
            "loss_commentary": loss_commentary.detach()
        }

def decode_corn_predictions(corn_logits: torch.Tensor) -> torch.Tensor:
    probs = torch.sigmoid(corn_logits)
    cum_probs = torch.cumprod(probs, dim=-1)
    return (cum_probs >= 0.5).sum(dim=-1)

star_counts = df_train["ordinal_rating"].value_counts().sort_index()
total_tr = len(df_train)
corn_w = torch.tensor(
    [math.sqrt(total_tr / (5.0 * max(star_counts.get(k, 1), 1))) for k in range(5)],
    dtype=torch.float32,
    device=DEVICE
)
print("CORN class weights:", corn_w.cpu().numpy().round(3))


## 8. Khởi tạo Mô hình và Vòng lặp Huấn luyện


In [ ]:
from contextlib import nullcontext

def get_amp_context(enabled: bool):
    if not enabled or DEVICE.type != "cuda":
        return nullcontext()
    try:
        return torch.amp.autocast("cuda", dtype=torch.float16, enabled=True)
    except Exception:
        return torch.cuda.amp.autocast(enabled=True)

def create_grad_scaler(enabled: bool):
    if not enabled or DEVICE.type != "cuda":
        try:
            return torch.amp.GradScaler("cuda", enabled=False)
        except Exception:
            return torch.cuda.amp.GradScaler(enabled=False)
    try:
        return torch.amp.GradScaler("cuda", enabled=True)
    except Exception:
        return torch.cuda.amp.GradScaler(enabled=True)

base_model = CRAM_ABSA_MultiAspectModel(cfg).to(DEVICE)
if NUM_GPUS > 1:
    print(f"Kích hoạt chế độ huấn luyện song song đa GPU (nn.DataParallel) trên {NUM_GPUS} GPUs!")
    model = nn.DataParallel(base_model, device_ids=list(range(NUM_GPUS)))
else:
    model = base_model

criterion = CRAMABSALoss(cfg, corn_w, pad_token_id=tokenizer.pad_token_id or 0).to(DEVICE)

raw_mod = model.module if isinstance(model, nn.DataParallel) else model
optimizer_grouped_parameters = [
    {"params": raw_mod.bert.parameters(), "lr": cfg.LR_BACKBONE, "weight_decay": cfg.WEIGHT_DECAY},
    {"params": [p for n, p in raw_mod.named_parameters() if not n.startswith("bert.")], "lr": cfg.LR_HEADS, "weight_decay": cfg.WEIGHT_DECAY}
]
optimizer = torch.optim.AdamW(optimizer_grouped_parameters)
total_steps = max(1, math.ceil(len(train_loader) / cfg.GRAD_ACCUM_STEPS) * cfg.EPOCHS)
warmup_steps = int(total_steps * cfg.WARMUP_RATIO)
scheduler = get_cosine_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
scaler = create_grad_scaler(cfg.USE_AMP_FP16)

@torch.no_grad()
def evaluate_cram_absa(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    all_star_preds, all_star_trues = [], []
    all_asp_preds, all_asp_trues = [], []
    rat_inter, rat_pred_sum, rat_true_sum = 0.0, 0.0, 0.0
    overlap_rates = []

    for batch in loader:
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        dense_features = batch["dense_features"].to(DEVICE, non_blocking=True)
        batch_gpu = {k: (v.to(DEVICE, non_blocking=True) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}

        with get_amp_context(cfg.USE_AMP_FP16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, dense_features=dense_features, commentary_target_ids=batch_gpu.get("commentary_target_ids"))
            losses = criterion(outputs, batch_gpu)

        total_loss += losses["total_loss"].item()
        star_preds = decode_corn_predictions(outputs["corn_logits"])
        all_star_preds.extend(star_preds.cpu().numpy())
        all_star_trues.extend(batch["ordinal_rating"].numpy())

        asp_preds = outputs["aspect_logits"].argmax(dim=-1)
        all_asp_preds.append(asp_preds.cpu().numpy())
        all_asp_trues.append(batch["aspect_sent_labels"].numpy())

        pred_mask = (outputs["rationale_probs"] >= 0.5).float()
        true_mask = batch_gpu["rationale_masks"]
        rat_inter += (pred_mask * true_mask).sum().item()
        rat_pred_sum += pred_mask.sum().item()
        rat_true_sum += true_mask.sum().item()

        active_asp_per_tok = pred_mask.sum(dim=1)
        leaked_toks = (active_asp_per_tok > 1.0).float().sum().item()
        active_toks = (active_asp_per_tok >= 1.0).float().sum().item()
        overlap_rates.append(leaked_toks / max(active_toks, 1.0))

    y_true_star = np.array(all_star_trues)
    y_pred_star = np.array(all_star_preds)
    acc_at_0 = accuracy_score(y_true_star, y_pred_star) * 100.0

    per_star_mae = [
        np.abs(y_true_star[y_true_star == k] - y_pred_star[y_true_star == k]).mean()
        for k in range(5) if np.any(y_true_star == k)
    ]
    macro_mae = float(np.mean(per_star_mae)) if per_star_mae else 0.0

    Y_asp_true = np.vstack(all_asp_trues)
    Y_asp_pred = np.vstack(all_asp_preds)
    aspect_f1s = [
        f1_score(Y_asp_true[:, a], Y_asp_pred[:, a], average="macro", zero_division=0) * 100.0
        for a in range(cfg.NUM_ASPECTS)
    ]
    macro_aspect_f1 = float(np.mean(aspect_f1s))
    rationale_f1 = (2.0 * rat_inter / max(rat_pred_sum + rat_true_sum, 1.0)) * 100.0
    leakage_rate = float(np.mean(overlap_rates)) * 100.0

    return {
        "val_loss": total_loss / max(len(loader), 1),
        "acc_at_0": acc_at_0,
        "macro_mae": macro_mae,
        "aspect_macro_f1": macro_aspect_f1,
        "rationale_f1": rationale_f1,
        "leakage_rate": leakage_rate,
        "per_aspect_f1": dict(zip(cfg.CANONICAL_ASPECTS, [round(x, 2) for x in aspect_f1s]))
    }

history = []
best_score = -float("inf")

print("Bắt đầu huấn luyện mô hình CRAM-ABSA...")
for epoch in range(1, cfg.EPOCHS + 1):
    model.train()
    ep_start = time.time()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    for step, batch in enumerate(train_loader, 1):
        input_ids = batch["input_ids"].to(DEVICE, non_blocking=True)
        attention_mask = batch["attention_mask"].to(DEVICE, non_blocking=True)
        dense_features = batch["dense_features"].to(DEVICE, non_blocking=True)
        batch_gpu = {k: (v.to(DEVICE, non_blocking=True) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}

        with get_amp_context(cfg.USE_AMP_FP16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, dense_features=dense_features, commentary_target_ids=batch_gpu.get("commentary_target_ids"))
            losses = criterion(outputs, batch_gpu)
            loss = losses["total_loss"] / cfg.GRAD_ACCUM_STEPS

        scaler.scale(loss).backward()
        running_loss += losses["total_loss"].item()

        if step % cfg.GRAD_ACCUM_STEPS == 0 or step == len(train_loader):
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.MAX_GRAD_NORM)
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

    val_metrics = evaluate_cram_absa(model, val_loader, criterion)
    ep_time = time.time() - ep_start
    vram_used = sum(torch.cuda.max_memory_allocated(i) / (1024 ** 3) for i in range(NUM_GPUS)) if NUM_GPUS > 0 else 0.0

    composite_score = val_metrics["aspect_macro_f1"] + val_metrics["acc_at_0"] - 20.0 * val_metrics["macro_mae"]
    is_best = composite_score > best_score
    if is_best:
        best_score = composite_score
        raw_state = (model.module if isinstance(model, nn.DataParallel) else model).state_dict()
        ckpt_payload = {
            "epoch": epoch,
            "model_name": "CRAM-ABSA v1 (modelv1.pt)",
            "model_state_dict": raw_state,
            "feat_mean": feat_mean,
            "feat_std": feat_std,
            "feature_cols_20": cfg.FEATURE_COLS_20,
            "canonical_aspects": cfg.CANONICAL_ASPECTS,
            "val_metrics": val_metrics
        }
        torch.save(ckpt_payload, cfg.MODEL_SAVE_PATH)
        if cfg.LOCAL_MODEL_SAVE_PATH.resolve() != cfg.MODEL_SAVE_PATH.resolve():
            shutil.copy2(cfg.MODEL_SAVE_PATH, cfg.LOCAL_MODEL_SAVE_PATH)

    epoch_record = {
        "epoch": epoch,
        "train_loss": round(running_loss / max(len(train_loader), 1), 4),
        **{k: (round(v, 4) if isinstance(v, float) else v) for k, v in val_metrics.items()},
        "vram_gb": round(vram_used, 2),
        "elapsed_s": round(ep_time, 1)
    }
    history.append(epoch_record)
    star_flag = "★ [BEST -> modelv1.pt]" if is_best else ""
    print(
        f"Epoch {epoch:02d}/{cfg.EPOCHS:02d} ({ep_time:.1f}s | VRAM: {vram_used:.2f}GB) | "
        f"Train Loss: {epoch_record['train_loss']:.4f} | Val Acc@0: {val_metrics['acc_at_0']:.2f}% | "
        f"6-Aspect F1: {val_metrics['aspect_macro_f1']:.2f}% | Macro-MAE: {val_metrics['macro_mae']:.4f} | "
        f"Rat-F1: {val_metrics['rationale_f1']:.2f}% | Leakage: {val_metrics['leakage_rate']:.2f}% {star_flag}"
    )

with open(cfg.HISTORY_SAVE_PATH, "w", encoding="utf-8") as f:
    json.dump(history, f, indent=2, ensure_ascii=False)
print(f"\nHoàn tất huấn luyện! File model đã được lưu với tên: {cfg.MODEL_SAVE_PATH}")


## 9. Suy luận và Sinh Văn bản Đánh giá 6 Khía cạnh


In [ ]:
@torch.no_grad()
def generate_structured_review_output(
    new_review_text: str,
    model: nn.Module,
    tokenizer,
    feat_mean: Optional[np.ndarray] = None,
    feat_std: Optional[np.ndarray] = None
) -> Dict[str, Any]:
    raw_model = model.module if isinstance(model, nn.DataParallel) else model
    raw_model.eval()

    raw_text = unicodedata.normalize("NFC", str(new_review_text).strip())
    raw_feat_20 = extract_20_features_for_new_review(raw_text, recency_weight=1.0)
    if feat_mean is not None and feat_std is not None:
        norm_feat_20 = (raw_feat_20 - feat_mean) / np.where(feat_std < 1e-6, 1.0, feat_std)
    else:
        norm_feat_20 = raw_feat_20

    enc = tokenizer(
        raw_text,
        max_length=cfg.MAX_SEQ_LEN,
        padding="max_length",
        truncation=True,
        return_tensors="pt"
    )
    input_ids = enc["input_ids"].to(DEVICE)
    attention_mask = enc["attention_mask"].to(DEVICE)
    dense_tensor = torch.tensor(norm_feat_20, dtype=torch.float32, device=DEVICE).unsqueeze(0)

    outputs = raw_model(input_ids=input_ids, attention_mask=attention_mask, dense_features=dense_tensor)
    pred_global_star = int(decode_corn_predictions(outputs["corn_logits"]).item()) + 1
    asp_probs = F.softmax(outputs["aspect_logits"].squeeze(0), dim=-1).cpu().numpy()
    asp_stars = decode_corn_predictions(outputs["aspect_star_logits"].squeeze(0)).cpu().numpy() + 1

    mentioned_aspects = []
    aspect_outputs = {}

    for a_idx, asp in enumerate(cfg.CANONICAL_ASPECTS):
        probs_4 = asp_probs[a_idx]
        pred_cls = int(np.argmax(probs_4))
        is_mentioned = (pred_cls != 0)

        if is_mentioned:
            mentioned_aspects.append(asp)
            star_val = int(asp_stars[a_idx])
            sent_label = ID_TO_SENTIMENT[pred_cls]
            gen_text = raw_model.generate_aspect_commentary(a_idx, outputs, input_ids, attention_mask, tokenizer)
            aspect_outputs[asp] = {
                "mentioned": True,
                "stars": star_val,
                "sentiment": sent_label,
                "text": f"{star_val}/5 ★ ({sent_label}) - {gen_text}"
            }
        else:
            aspect_outputs[asp] = {
                "mentioned": False,
                "stars": None,
                "sentiment": "Not Mentioned",
                "text": ""
            }

    aspect_list_str = ", ".join(mentioned_aspects) if mentioned_aspects else "None"
    lines = [f"- Aspect: {aspect_list_str}"]
    for asp in cfg.CANONICAL_ASPECTS:
        info = aspect_outputs[asp]
        if info["mentioned"]:
            lines.append(f"- {asp}: {info['text']}")
        else:
            lines.append(f"- {asp}:")

    return {
        "Aspect": mentioned_aspects,
        "overall_star": pred_global_star,
        "aspects": aspect_outputs,
        "formatted_text": "\n".join(lines)
    }

def load_trained_model_and_predict(new_review_text: str, checkpoint_path: Optional[str] = None) -> str:
    ckpt_file = Path(checkpoint_path) if checkpoint_path else cfg.MODEL_SAVE_PATH
    if not ckpt_file.exists() and cfg.LOCAL_MODEL_SAVE_PATH.exists():
        ckpt_file = cfg.LOCAL_MODEL_SAVE_PATH
    if not ckpt_file.exists():
        raise FileNotFoundError(f"Không tìm thấy file mô hình tại: {ckpt_file}")

    ckpt = torch.load(ckpt_file, map_location=DEVICE, weights_only=False)
    infer_model = CRAM_ABSA_MultiAspectModel(cfg).to(DEVICE)
    infer_model.load_state_dict(ckpt["model_state_dict"], strict=False)

    f_mean = ckpt.get("feat_mean", globals().get("feat_mean", None))
    f_std = ckpt.get("feat_std", globals().get("feat_std", None))
    tok = globals().get("tokenizer", None)
    if tok is None:
        tok = AutoTokenizer.from_pretrained(cfg.BACKBONE_NAME)

    res = generate_structured_review_output(
        new_review_text=new_review_text,
        model=infer_model,
        tokenizer=tok,
        feat_mean=f_mean,
        feat_std=f_std
    )
    return res["formatted_text"]
